# STATISTICAL LEARNING FINAL PROJECT
## Raissa Kitenge

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('/content/drive/MyDrive/feature_engineered_real_estate.csv')
df.head()

# Step 1

In [ ]:
# Check data
print(df.shape)
print(df.columns)
print(df.isnull().sum())

In [ ]:
# Define response and predictors
y_col = "PriceRatio"

x_cols = [
    "TotalFinishedArea",
    "LivingUnits",
    "TotalAppraisedValue",
    "SaleYear",
    "SaleMonth",
    "AssrLandUse_APT FOUR",
    "AssrLandUse_CONDOMINIMUM",
    "AssrLandUse_MULTI DWLG",
    "AssrLandUse_THREE FAMILY",
    "AssrLandUse_TWO FAMILY"
]

In [ ]:
# Split into training and test sets
df_train, df_test = train_test_split(df, test_size=0.2, random_state=6243)

print(df_train.shape)
print(df_test.shape)

In [ ]:
# Standardize only the continuous predictors
cont_cols = [
    "TotalFinishedArea",
    "LivingUnits",
    "TotalAppraisedValue",
    "SaleYear",
    "SaleMonth"
]

scaler = StandardScaler()

df_train = df_train.copy()
df_test = df_test.copy()

df_train[cont_cols] = scaler.fit_transform(df_train[cont_cols])
df_test[cont_cols] = scaler.transform(df_test[cont_cols])

df_train.head()

# Step 2

In [ ]:
with pm.Model() as bayes_model:
    # Priors
    intercept = pm.StudentT("Intercept", nu=4, mu=0, sigma=10)

    w_TotalFinishedArea   = pm.Normal("w_TotalFinishedArea", mu=0, sigma=10)
    w_LivingUnits         = pm.Normal("w_LivingUnits", mu=0, sigma=10)
    w_TotalAppraisedValue = pm.Normal("w_TotalAppraisedValue", mu=0, sigma=10)
    w_SaleYear            = pm.Normal("w_SaleYear", mu=0, sigma=10)
    w_SaleMonth           = pm.Normal("w_SaleMonth", mu=0, sigma=10)

    w_APT_FOUR            = pm.Normal("w_APT_FOUR", mu=0, sigma=10)
    w_CONDOMINIMUM        = pm.Normal("w_CONDOMINIMUM", mu=0, sigma=10)
    w_MULTI_DWLG          = pm.Normal("w_MULTI_DWLG", mu=0, sigma=10)
    w_THREE_FAMILY        = pm.Normal("w_THREE_FAMILY", mu=0, sigma=10)
    w_TWO_FAMILY          = pm.Normal("w_TWO_FAMILY", mu=0, sigma=10)

    sigma = pm.HalfNormal("sigma", sigma=1)

    # Linear predictor
    mu = (
        intercept
        + w_TotalFinishedArea   * df_train["TotalFinishedArea"]
        + w_LivingUnits         * df_train["LivingUnits"]
        + w_TotalAppraisedValue * df_train["TotalAppraisedValue"]
        + w_SaleYear            * df_train["SaleYear"]
        + w_SaleMonth           * df_train["SaleMonth"]
        + w_APT_FOUR            * df_train["AssrLandUse_APT FOUR"]
        + w_CONDOMINIMUM        * df_train["AssrLandUse_CONDOMINIMUM"]
        + w_MULTI_DWLG          * df_train["AssrLandUse_MULTI DWLG"]
        + w_THREE_FAMILY        * df_train["AssrLandUse_THREE FAMILY"]
        + w_TWO_FAMILY          * df_train["AssrLandUse_TWO FAMILY"]
    )

    # Likelihood
    y_obs = pm.Normal("PriceRatio", mu=mu, sigma=sigma, observed=df_train[y_col])

    # Posterior sampling
    trace_bayes = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        random_seed=6243,
        target_accept=0.9,
        return_inferencedata=True
    )

# Step 3

In [ ]:
# Posterior summaries

# Summary table
summary = az.summary(trace_bayes, round_to=4)
print(summary)

In [ ]:
az.plot_trace(
    trace_bayes,
    var_names=[
        "Intercept", "sigma",
        "w_TotalFinishedArea", "w_LivingUnits", "w_TotalAppraisedValue",
        "w_SaleYear", "w_SaleMonth",
        "w_APT_FOUR", "w_CONDOMINIMUM", "w_MULTI_DWLG",
        "w_THREE_FAMILY", "w_TWO_FAMILY"
    ],
    figsize=(14, 22)
)
plt.tight_layout()
plt.show()

In [ ]:
!jupyter nbconvert --to pdf '/content/drive/MyDrive/Colab Notebooks/07_bayesian_model.ipynb'